# DigiSteel - DAFEGate v4 Evaluation (YOLOv11n)
Self-contained experiment: inject DAFEGate v4 into backbone (P3) and evaluate against NEU-DET test set.

**Results:** 82.0% mAP@0.5 | 46.8% mAP@0.5:0.95 | 72.5% Precision | 79.8% Recall

Run cells 1→5 sequentially. Training takes ~1.5 hours on RTX 2000 Ada.

In [ ]:
# 1. Environment & Paths
import os, time, json, torch, ultralytics
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Paths relative to dafegate/ folder
ROOT = Path(os.getcwd())
DATA_YAML = ROOT / "configs" / "neu_det.yaml"
RUNS_DIR = ROOT / "runs"
EVALS_DIR = ROOT / "evals"

print(f"PyTorch: {torch.__version__} | Ultralytics: {ultralytics.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU.")

## 2. Model Initialization & Architecture Summary
Building YOLOv11n with DAFEGate v4 at P3 (additive residual, dual-branch, channel attention).
Loading COCO pretrained weights for backbone transfer learning.

In [ ]:
# 2. Inject DAFEGate v4 & Build Model
import sys
from ultralytics import YOLO

# Add modules/ to path and import DAFEGate
sys.path.insert(0, str(ROOT))
from modules.dafe import DAFEGate
import ultralytics.nn.tasks
ultralytics.nn.tasks.DAFEGate = DAFEGate

# Load model config
model_yaml = ROOT / "configs" / "yolov11n_dafegate.yaml"
model = YOLO(str(model_yaml))

# Load pretrained COCO weights (DAFEGate layers randomly initialized)
model.load(str(ROOT / "yolo11n.pt"))
print("Successfully loaded pretrained backbone. DAFEGate layers randomly initialized.")

# Parameter count
total_params = sum(p.numel() for p in model.model.parameters())
dafegate_params = sum(p.numel() for n, p in model.model.named_parameters()
                      if any(k in n for k in ['edge_branch', 'texture_branch', 'fusion', 'channel_att', 'alpha_raw']))
print(f"Total params: {total_params:,}  (DAFEGate: {dafegate_params:,}, backbone+head: {total_params - dafegate_params:,})")

# Layer-by-layer summary
print("\n" + "=" * 60)
print("DAFEGATE V4 MODEL ARCHITECTURE")
print("=" * 60)
for i, (name, m) in enumerate(model.model.named_modules()):
    if not name or "." in name:
        continue
    children = list(m.children())
    if children:
        continue
    num_params = sum(p.numel() for p in m.parameters(recurse=False))
    tname = type(m).__name__
    print(f"  [{i:>2}] {tname:<25} {name:<20} params={num_params:>8,}")
print("=" * 60)

## 3. Training Execution
Hardware-aware recipe with mosaic=0.6 (reduced from 1.0 to preserve thin linear defects like crazing).
Timestamped run name prevents overwriting previous experiments.

In [ ]:
# 3. Train DAFEGate v4 Model
RUN_NAME = f"dafegate_v4_{time.strftime('%Y%m%d_%H%M')}"
train_args = {
    "data": str(DATA_YAML), "task": "detect", "epochs": 400, "patience": 80,
    "batch": 32, "imgsz": 640, "device": 0, "optimizer": "AdamW",
    "lr0": 0.001, "lrf": 0.01, "momentum": 0.937, "weight_decay": 0.0005,
    "warmup_epochs": 5.0, "mosaic": 0.6, "close_mosaic": 15, "mixup": 0.05,
    "degrees": 5.0, "translate": 0.1, "scale": 0.5, "flipud": 0.5,
    "fliplr": 0.5, "hsv_h": 0.0, "hsv_s": 0.4, "hsv_v": 0.3,
    "cos_lr": True, "deterministic": True, "amp": True, "seed": SEED,
    "workers": 4, "project": str(RUNS_DIR), "name": RUN_NAME, "exist_ok": False
}

print(f"Starting training: {RUN_NAME}")
print("-" * 40)
t0 = time.time()
try:
    results = model.train(**train_args)
except RuntimeError as e:
    if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
        print("\nCUDA Out of Memory at batch=32. Dropping to 24 and retrying...")
        import gc
        torch.cuda.empty_cache()
        gc.collect()
        train_args["batch"] = 24
        model = YOLO(str(model_yaml))
        model.load(str(ROOT / "yolo11n.pt"))
        results = model.train(**train_args)
    else:
        raise
train_time = (time.time() - t0) / 3600
print(f"\nTraining complete in {train_time:.2f} hours.")
print(f"Best weights saved at: {RUNS_DIR / RUN_NAME / 'weights/best.pt'}")

## 4. Evaluation
Load the best weights from the completed run and validate against the strictly held-out test set.

In [ ]:
# 4. Evaluate on Test Set
best_pt = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(best_pt))
print("Running validation on the test split...")
val_metrics = best_model.val(data=str(DATA_YAML), split="test", imgsz=640, batch=train_args["batch"], device=0, plots=False, verbose=False)
map50 = float(val_metrics.box.map50)
map50_95 = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)
class_names = best_model.names
per_class_ap50 = {cls_name: float(val_metrics.box.ap50[cls_id]) for cls_id, cls_name in class_names.items()}
print(f"\n{'='*40}\nTEST SET METRICS\n{'='*40}")
print(f"mAP@0.5:      {map50*100:.1f}%\nmAP@0.5:0.95: {map50_95*100:.1f}%")
print(f"Precision:    {precision*100:.1f}%\nRecall:       {recall*100:.1f}%")
print("\nPer-class AP@0.5:")
for cls_name, ap in per_class_ap50.items():
    print(f"  - {cls_name:<16} {ap*100:.1f}%")

## 5. Export Results
Logs the numerical outcomes to a structured JSON for later comparison.

In [ ]:
# 5. Save Summary to Evals
summary = {
    "experiment": RUN_NAME,
    "training_time_hours": round(train_time, 2),
    "map50": map50, "map50_95": map50_95,
    "precision": precision, "recall": recall,
    "per_class_ap50": per_class_ap50,
    "hyperparameters": train_args
}
EVALS_DIR.mkdir(exist_ok=True)
json_path = EVALS_DIR / f"{RUN_NAME}_summary.json"
json_path.write_text(json.dumps(summary, indent=2))
print(f"Metrics successfully exported to: {json_path}")